In [ ]:
!pip install torch torchvision scikit-learn opencv-python matplotlib tqdm segmentation-models-pytorch scipy --quiet

In [ ]:
import os
import json
import random
import shutil
from glob import glob

import cv2
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

import segmentation_models_pytorch as smp
import timm

from tqdm.auto import tqdm
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import (
    accuracy_score, recall_score, precision_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix,
    precision_recall_curve, auc, matthews_corrcoef
)
from scipy.spatial.distance import directed_hausdorff

from torch.amp import GradScaler, autocast

# Mounting Drive
try:
    from google.colab import drive
    if not os.path.isdir('/content/drive') or len(os.listdir('/content/drive')) == 0:
        drive.mount('/content/drive', force_remount=True)
except ImportError:
    pass

# 2. Paths, Reproducibility & Preprocessing

# 2.1 Reproducibility & Config
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
USE_AMP = (DEVICE == 'cuda')
print(f"Using device: {DEVICE}")

# 2.2 Dataset and output root (adjust path as needed)
DS_ROOT = '/content/drive/MyDrive/Dataset2'
OUT_ROOT = '/content/drive/MyDrive/Thyroid_Results_SWIN_CSASN'
os.makedirs(OUT_ROOT, exist_ok=True)

# 2.3 Prepare folders for preprocessed images + masks
IMG_OUT = os.path.join(DS_ROOT, 'preprocessed_images')
MSK_OUT = os.path.join(DS_ROOT, 'preprocessed_masks')
CLASSES = ['Benign', 'Malignant']

for base in [IMG_OUT, MSK_OUT]:
    for cls in CLASSES:
        os.makedirs(os.path.join(base, cls), exist_ok=True)

# 2.4 Gather all image paths + labels
paths, labels = [], []
for idx, cls in enumerate(CLASSES):
    folder = os.path.join(DS_ROOT, cls)
    if not os.path.exists(folder):
        print(f"Warning: Folder {folder} does not exist")
        continue
    for ext in ('*.Jpg'):
        for fp in glob(os.path.join(folder, ext)):
            paths.append(fp)
            labels.append(idx)
print("Found total images:", len(paths))
if len(paths) == 0:
    raise ValueError(f"No images found under {DS_ROOT}. Ensure 'Benign'/'Malignant' subdirs exist.")

print("Class distribution:")
for i, cls in enumerate(CLASSES):
    count = labels.count(i)
    print(f"{cls}: {count} images ({count/len(labels)*100:.1f}%)")

# --- EDA #1: Combined 2×2 Figure ---
## Build DataFrame of basic file info
df = pd.DataFrame({'path': paths, 'label': labels})
df['class'] = df['label'].map({0: 'Benign', 1: 'Malignant'})
dims = [cv2.imread(p).shape[:2][::-1] for p in df['path']]  # (width,height)
df['width'], df['height'] = zip(*dims)
df['size_mb'] = [os.path.getsize(p)/1e6 for p in df['path']]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
counts = df['class'].value_counts()

# Pie
axes[0,0].pie(counts, labels=counts.index, autopct='%1.1f%%',
              colors=['#4CAF50','#F44336'], explode=(0.05,0))
axes[0,0].set_title('Class Distribution (Pie Chart)')

# Bar
axes[0,1].bar(counts.index, counts.values,
              color=['#4CAF50','#F44336'], edgecolor='k')
for i, v in enumerate(counts.values):
    axes[0,1].text(i, v+5, str(v), ha='center', fontweight='bold')
axes[0,1].set_ylabel('Number of Images')
axes[0,1].set_title('Class Distribution (Bar Chart)')

# Scatter dims
for cls, color in zip(['Benign','Malignant'], ['#4CAF50','#F44336']):
    subset = df[df['class']==cls]
    axes[1,0].scatter(subset['width'], subset['height'],
                      label=cls, alpha=0.7, s=40, edgecolor='k')
axes[1,0].set_xlabel('Width (pixels)')
axes[1,0].set_ylabel('Height (pixels)')
axes[1,0].set_title('Image Dimensions Distribution')
axes[1,0].legend()

# Boxplot file size
sns.boxplot(x='class', y='size_mb', hue='class', data=df,
            palette=['#4CAF50','#F44336'], ax=axes[1,1], legend=False)
axes[1,1].set_ylabel('File Size (MB)')
axes[1,1].set_title('File Size Distribution by Class')

plt.tight_layout()
plt.show()

# 2.5 Split into 70% train / 15% val / 15% test (stratified)
tmp_p, test_p, tmp_l, test_l = train_test_split(
    paths, labels, stratify=labels, test_size=0.15, random_state=seed
)
train_p, val_p, train_l, val_l = train_test_split(
    tmp_p, tmp_l, stratify=tmp_l, test_size=0.1765, random_state=seed
)
print("Dataset splits:")
print(f"  Train:      {len(train_p)} images")
print(f"  Validation: {len(val_p)} images")
print(f"  Test:       {len(test_p)} images")

# --- EDA #2: Train/Val/Test Split Visualization ---
def count_by_class(ps, ls):
    df_ = pd.DataFrame({'p': ps, 'l': ls})
    vc = df_['l'].value_counts().reindex([0,1]).fillna(0).values
    return vc.astype(int)

train_cls = count_by_class(train_p, train_l)
val_cls   = count_by_class(val_p, val_l)
test_cls  = count_by_class(test_p, test_l)
classes   = ['Benign','Malignant']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
bar_bottom = np.zeros(3)
for idx, cls in enumerate(classes):
    vals = [train_cls[idx], val_cls[idx], test_cls[idx]]
    color = '#4CAF50' if cls=='Benign' else '#F44336'
    ax1.bar(['Train','Validation','Test'], vals,
            bottom=bar_bottom, label=cls, color=color)
    bar_bottom += vals
for i, tot in enumerate(bar_bottom):
    ax1.text(i, tot+10, str(int(tot)), ha='center', fontweight='bold')
ax1.set_ylabel('Number of Images')
ax1.set_title('Dataset Split Distribution')
ax1.legend()

sizes = [len(train_p), len(val_p), len(test_p)]
ax2.pie(sizes, labels=['Train','Val','Test'], autopct='%1.1f%%',
        colors=['#2196F3','#FF9800','#9C27B0'], startangle=140)
ax2.set_title('Split Percentages')

plt.tight_layout()
plt.show()

# 2.6 Preprocess: Resize to 224×224 and extract Otsu mask
print("Starting preprocessing...")
processed_count = 0
for pth, lbl in tqdm(zip(paths, labels), total=len(paths), desc='Preprocessing'):
    out_img = os.path.join(IMG_OUT, CLASSES[lbl], os.path.basename(pth))
    out_msk = os.path.join(MSK_OUT, CLASSES[lbl], os.path.basename(pth))
    if os.path.exists(out_img) and os.path.exists(out_msk):
        processed_count += 1
        continue
    img = cv2.imread(pth)
    if img is None:
        print(f"Warning: Could not read image {pth}")
        continue
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    _, mask = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    try:
        cv2.imwrite(out_img, cv2.resize(rgb, (224, 224)))
        cv2.imwrite(out_msk, cv2.resize(mask, (224, 224)))
        processed_count += 1
    except Exception as e:
        print(f"Error processing {pth}: {e}")
print(f"Preprocessing completed. Processed {processed_count}/{len(paths)} images.")

# ----------------------------
# VISUALIZATION: Raw Images Before Preprocessing
# ----------------------------
if len(paths) > 0:
    sample_paths = random.sample(paths, min(3, len(paths)))
    plt.figure(figsize=(12, 4))
    for i, p in enumerate(sample_paths):
        img = cv2.imread(p)
        if img is not None:
            img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            plt.subplot(1, 3, i + 1)
            plt.imshow(img_rgb)
            plt.title(f"Raw: {os.path.basename(p)}")
            plt.axis('off')
    plt.tight_layout()
    plt.show()

# ----------------------------
# VISUALIZATION: Preprocessed Images and Masks
# ----------------------------
if len(paths) > 0:
    plt.figure(figsize=(12, 8))
    for i, p in enumerate(sample_paths):
        lbl = labels[paths.index(p)]
        fn = os.path.basename(p)
        pre_img_path = os.path.join(IMG_OUT, CLASSES[lbl], fn)
        pre_msk_path = os.path.join(MSK_OUT, CLASSES[lbl], fn)
        if os.path.exists(pre_img_path) and os.path.exists(pre_msk_path):
            pre_img = cv2.imread(pre_img_path)
            pre_msk = cv2.imread(pre_msk_path, cv2.IMREAD_GRAYSCALE)
            if pre_img is not None:
                pre_img = cv2.cvtColor(pre_img, cv2.COLOR_BGR2RGB)
                plt.subplot(2, 3, i + 1)
                plt.imshow(pre_img)
                plt.title(f"Preprocessed: {fn}")
                plt.axis('off')
            if pre_msk is not None:
                plt.subplot(2, 3, i + 4)
                plt.imshow(pre_msk, cmap='gray')
                plt.title(f"Mask: {fn}")
                plt.axis('off')
    plt.tight_layout()
    plt.show()

# --- EDA #4: Processed Images, Masks, Overlays Grid ---
sample = random.sample(list(zip(paths, labels)), k=min(6, len(paths)))
fig, axes = plt.subplots(3, len(sample), figsize=(18, 9))
for i, (p, lbl) in enumerate(sample):
    fn = os.path.basename(p)
    im_p = os.path.join(IMG_OUT, CLASSES[lbl], fn)
    m_p  = os.path.join(MSK_OUT, CLASSES[lbl], fn)
    img = cv2.cvtColor(cv2.imread(im_p), cv2.COLOR_BGR2RGB)
    m   = cv2.imread(m_p, cv2.IMREAD_GRAYSCALE) / 255.0
    overlay = (img * 0.6 + np.dstack([m]*3)*255*0.4).astype(np.uint8)

    axes[0,i].imshow(img);        axes[0,i].axis('off'); axes[0,i].set_title('Processed')
    axes[1,i].imshow(m, cmap='gray'); axes[1,i].axis('off'); axes[1,i].set_title('Mask')
    axes[2,i].imshow(overlay);    axes[2,i].axis('off'); axes[2,i].set_title('Overlay')
plt.suptitle('After Preprocessing: Processed Images, Masks, and Overlays', fontsize=18, y=1.02)
plt.tight_layout()
plt.show()

# 3. Dataset & DataLoader
MEAN = np.array([0.485, 0.456, 0.406, 0.5], dtype=np.float32)
STD  = np.array([0.229, 0.224, 0.225, 0.2], dtype=np.float32)

class ThyroidDataset(Dataset):
    def __init__(self, paths, labels, augment=False):
        self.paths = paths
        self.labels = labels
        self.augment = augment

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        p = self.paths[idx]; l = self.labels[idx]; fn = os.path.basename(p)
        img_path = os.path.join(IMG_OUT, CLASSES[l], fn)
        img = cv2.imread(img_path) if os.path.exists(img_path) else None
        if img is None:
            raw = cv2.imread(p)
            img = cv2.resize(cv2.cvtColor(raw, cv2.COLOR_BGR2RGB), (224,224))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        mask_path = os.path.join(MSK_OUT, CLASSES[l], fn)
        m = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE) if os.path.exists(mask_path) else None
        if m is None:
            raw = cv2.imread(p)
            gray = cv2.cvtColor(raw, cv2.COLOR_BGR2GRAY)
            _, mask_full = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
            m = cv2.resize(mask_full, (224,224))

        if self.augment:
            if random.random() < 0.5:
                img = np.fliplr(img).copy(); m = np.fliplr(m).copy()
            if random.random() < 0.5:
                angle = random.uniform(-15,15)
                h,w = img.shape[:2]
                M = cv2.getRotationMatrix2D((w//2,h//2), angle,1.0)
                img = cv2.warpAffine(img, M, (w,h), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_REFLECT)
                m   = cv2.warpAffine(m,   M, (w,h), flags=cv2.INTER_NEAREST, borderMode=cv2.BORDER_REFLECT)
            if random.random() < 0.5:
                alpha = random.uniform(0.9,1.1); beta = random.uniform(-10,10)
                img = np.clip(alpha*img + beta, 0,255).astype(np.uint8)

        img = img.astype(np.float32)/255.0
        m   = m.astype(np.float32)/255.0
        x_np = np.dstack([img,m])
        x_np = (x_np - MEAN)/STD
        x = torch.tensor(x_np, dtype=torch.float32).permute(2,0,1)
        y = torch.tensor(l, dtype=torch.long)
        mask_tensor = torch.tensor(m, dtype=torch.float32)
        return x, y, mask_tensor

dl_kwargs = {
    'batch_size': 8,
    'num_workers': 2,
    'pin_memory': torch.cuda.is_available(),
    'persistent_workers': torch.cuda.is_available()
}

# Compute class weights
train_counts = np.bincount(train_l)
class_weights = torch.tensor(
    [len(train_l)/(2.0*c) for c in train_counts],
    dtype=torch.float32
).to(DEVICE)
print(f"Class weights: {class_weights}")

train_loader = DataLoader(ThyroidDataset(train_p, train_l, augment=True), shuffle=True, **dl_kwargs)
val_loader   = DataLoader(ThyroidDataset(val_p, val_l, augment=False), shuffle=False, **dl_kwargs)
test_loader  = DataLoader(ThyroidDataset(test_p, test_l, augment=False), shuffle=False, **dl_kwargs)

# 4. Model Architectures
class SegSwinUNet(nn.Module):
    def __init__(self):
        super().__init__()
        try:
            self.unet = smp.Unet(encoder_name='mit_b5', encoder_weights='imagenet',
                                 in_channels=3, classes=1, decoder_attention_type='scse')
        except:
            self.unet = smp.Unet(encoder_name='resnet34', encoder_weights='imagenet',
                                 in_channels=3, classes=1, decoder_attention_type='scse')
    def forward(self, x):
      return self.unet(x)

class ChannelSpatialAttention(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.se = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Flatten(),
            nn.Linear(channels, max(channels//16,1), bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(max(channels//16,1), channels, bias=False),
            nn.Sigmoid()
        )
        self.spatial = nn.Sequential(
            nn.Conv2d(channels, 1, kernel_size=7, padding=3, bias=False),
            nn.Sigmoid()
        )
    def forward(self, x):
        b,c,h,w = x.size()
        se_w = self.se(x).view(b,c,1,1)
        x_se = x * se_w
        sa_w = self.spatial(x_se)
        return x_se * sa_w

class CSASNClassifier(nn.Module):
    def __init__(self, pretrained=True):
        super().__init__()
        try:
            self.effnet = timm.create_model('efficientnet_b5', pretrained=pretrained)
            eff_features = self.effnet.num_features
        except:
            self.effnet = timm.create_model('efficientnet_b0', pretrained=pretrained)
            eff_features = self.effnet.num_features
        self.effnet.classifier = nn.Identity()
        self.reduce_eff = nn.Conv2d(eff_features, 512, kernel_size=1, bias=False)
        self.cs_eff    = ChannelSpatialAttention(512)

        try:
            self.vit = timm.create_model('vit_base_patch16_224', pretrained=pretrained)
            vit_features = self.vit.num_features
        except:
            self.vit = timm.create_model('vit_small_patch16_224', pretrained=pretrained)
            vit_features = self.vit.num_features
        self.vit.head = nn.Identity()
        self.cs_vit   = ChannelSpatialAttention(vit_features)

        self.fusion_fc = nn.Sequential(
            nn.Linear(512 + vit_features, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),
            nn.Linear(512, 2)
        )

    def forward(self, x):
        b = x.size(0)
        # EfficientNet branch
        eff_feats = self.effnet.forward_features(x)
        eff_red   = self.reduce_eff(eff_feats)
        eff_att   = self.cs_eff(eff_red)
        eff_vec   = F.adaptive_avg_pool2d(eff_att,1).view(b,512)
        # ViT branch
        vit_feats = self.vit.forward_features(x)
        if vit_feats.ndim == 2:
            vit_cls = vit_feats
        else:
            vit_cls = vit_feats[:,0,:]
        vit_rs  = vit_cls.unsqueeze(-1).unsqueeze(-1)
        vit_att = self.cs_vit(vit_rs).view(b, vit_cls.size(1))
        fused   = torch.cat([eff_vec, vit_att], dim=1)
        return self.fusion_fc(fused)

class HybridSwinCSASN(nn.Module):
    def __init__(self):
        super().__init__()
        self.seg = SegSwinUNet()
        self.cls = CSASNClassifier(pretrained=True)
    def forward(self, x):
        rgb = x[:, :3]
        seg_logits = self.seg(rgb)
        cls_logits = self.cls(rgb)
        return seg_logits, cls_logits

# 5. Losses & Optimizers
class DiceBCELoss(nn.Module):
    def __init__(self, smooth=1e-5):
        super().__init__()
        self.smooth = smooth
        self.bce    = nn.BCEWithLogitsLoss()
    def forward(self, logits, targets):
        if targets.ndim == 3: targets = targets.unsqueeze(1)
        bce_loss = self.bce(logits, targets)
        pred_flat = torch.sigmoid(logits).view(logits.size(0), -1)
        tgt_flat  = targets.view(targets.size(0), -1)
        inter     = (pred_flat * tgt_flat).sum(dim=1)
        dice_score= (2*inter + self.smooth)/(pred_flat.sum(dim=1)+tgt_flat.sum(dim=1)+self.smooth)
        dice_loss = 1 - dice_score.mean()
        return bce_loss + dice_loss

seg_loss_fn = DiceBCELoss()
cls_loss_fn = nn.CrossEntropyLoss(weight=class_weights)

# 6. Training with 5-Fold CV
def train_folds(models_save_dir=OUT_ROOT, num_epochs=30, patience_init=5):
    dl_kwargs_small = dl_kwargs
    models = []
    kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)

    for fold, (tr_idx, va_idx) in enumerate(kf.split(train_p, train_l), start=1):
        print(f"\n=== Fold {fold} ===")
        fold_dir   = os.path.join(models_save_dir, f'results_fold{fold}')
        model_path = os.path.join(fold_dir, 'best_model.pth')
        if os.path.exists(model_path):
            print(f"Loading existing fold {fold}")
            model = HybridSwinCSASN().to(DEVICE)
            model.load_state_dict(torch.load(model_path, map_location=DEVICE))
            models.append(model)
            continue

        shutil.rmtree(fold_dir, ignore_errors=True)
        os.makedirs(fold_dir, exist_ok=True)
        tr_paths  = [train_p[i] for i in tr_idx]
        tr_labels = [train_l[i] for i in tr_idx]
        va_paths  = [train_p[i] for i in va_idx]
        va_labels = [train_l[i] for i in va_idx]

        tr_loader = DataLoader(ThyroidDataset(tr_paths, tr_labels, augment=True),
                               shuffle=True, **dl_kwargs_small)
        va_loader = DataLoader(ThyroidDataset(va_paths, va_labels, augment=False),
                               shuffle=False, **dl_kwargs_small)

        model     = HybridSwinCSASN().to(DEVICE)
        optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-2)
        scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)
        scaler    = GradScaler() if USE_AMP else None

        best_acc, patience, best_state = 0.0, patience_init, None
        tr_losses, val_accs = [], []

        for epoch in range(1, num_epochs+1):
            model.train()
            running_loss = 0.0
            for xb, yb, mb in tqdm(tr_loader, desc=f"Fold{fold} E{epoch} train", leave=False):
                xb, yb, mb = xb.to(DEVICE), yb.to(DEVICE), mb.to(DEVICE)
                optimizer.zero_grad()
                ctx = autocast("cuda") if USE_AMP else autocast("cpu")
                with ctx:
                    seg_logits, cls_logits = model(xb)
                    loss = seg_loss_fn(seg_logits, mb) + cls_loss_fn(cls_logits, yb)
                if USE_AMP:
                    scaler.scale(loss).backward()
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
                    optimizer.step()
                running_loss += loss.item()
            scheduler.step()
            tr_losses.append(running_loss/len(tr_loader))

            # Validation
            model.eval()
            val_probs, val_truths = [], []
            with torch.no_grad():
                for xb, yb, _ in tqdm(va_loader, desc=f"Fold{fold} E{epoch} val", leave=False):
                    xb = xb.to(DEVICE)
                    ctx = autocast("cuda") if USE_AMP else autocast("cpu")
                    with ctx:
                        _, out = model(xb)
                    probs = torch.softmax(out, dim=1)[:,1]
                    val_probs.extend(probs.cpu().tolist())
                    val_truths.extend(yb.tolist())

            if val_probs:
                preds = [1 if p>=0.5 else 0 for p in val_probs]
                acc   = accuracy_score(val_truths, preds)
                val_accs.append(acc)
                print(f"Epoch {epoch} Val Acc: {acc:.4f}")
                if acc > best_acc:
                    best_acc, best_state, patience = acc, model.state_dict().copy(), patience_init
                else:
                    patience -= 1
                    if patience == 0:
                        print(f"Early stopping at epoch {epoch}")
                        break

        # Save best for this fold
        if best_state:
            best_model = HybridSwinCSASN().to(DEVICE)
            best_model.load_state_dict(best_state)
            best_model.eval()
            torch.save(best_state, model_path)
            models.append(best_model)

            # Confusion matrix on validation
            fold_truths, fold_preds = [], []
            with torch.no_grad():
                for xb, yb, _ in va_loader:
                    xb = xb.to(DEVICE)
                    ctx = autocast("cuda") if USE_AMP else autocast("cpu")
                    with ctx:
                        _, out_cls = best_model(xb)
                    probs = torch.softmax(out_cls, dim=1)[:,1].cpu().numpy()
                    preds = (probs >= 0.5).astype(int)
                    fold_preds.extend(preds.tolist())
                    fold_truths.extend(yb.tolist())
            cm = confusion_matrix(fold_truths, fold_preds)
            plt.figure()
            plt.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
            plt.title(f'Fold {fold} Confusion Matrix')
            plt.colorbar()
            plt.xticks([0,1],CLASSES)
            plt.yticks([0,1],CLASSES)
            plt.xlabel('Predicted'); plt.ylabel('True')
            plt.show()

            # Save training curves
            plt.figure()
            plt.plot(tr_losses, label='Train Loss')
            plt.plot(val_accs, label='Val Acc')
            plt.xlabel('Epoch'); plt.ylabel('Loss/Acc')
            plt.legend()
            plt.title(f'Fold {fold} Curves')
            plt.savefig(os.path.join(fold_dir,'training_curves.png'))
            plt.show()

            fold_results = {
                'fold': fold,
                'best_val_acc': best_acc,
                'train_losses': tr_losses,
                'val_accuracies': val_accs,
                'num_train': len(tr_paths),
                'num_val': len(va_paths)
            }
            with open(os.path.join(fold_dir, 'fold_results.json'), 'w') as f:
                json.dump(fold_results, f, indent=2)

    return models

# Train all folds
print("Starting 5-fold cross-validation training...")
cv_models = train_folds(OUT_ROOT, num_epochs=30, patience_init=5)
print(f"Training completed. {len(cv_models)} models trained.")

# 7. Ensemble Evaluation Functions
def ensemble_predict(models, loader, device=DEVICE):
    """Get ensemble predictions from multiple models"""
    all_probs, all_truths, all_seg_preds, all_seg_truths = [], [], [], []

    for model in models:
        model.eval()

    with torch.no_grad():
        for xb, yb, mb in tqdm(loader, desc="Ensemble prediction"):
            xb, yb, mb = xb.to(device), yb.to(device), mb.to(device)

            # Collect predictions from all models
            cls_probs_batch = []
            seg_preds_batch = []

            for model in models:
                ctx = autocast("cuda") if USE_AMP else autocast("cpu")
                with ctx:
                    seg_logits, cls_logits = model(xb)

                # Classification probabilities
                cls_probs = torch.softmax(cls_logits, dim=1)[:, 1]
                cls_probs_batch.append(cls_probs.cpu())

                # Segmentation predictions
                seg_preds = torch.sigmoid(seg_logits).squeeze(1)
                seg_preds_batch.append(seg_preds.cpu())

            # Average ensemble predictions
            avg_cls_probs = torch.stack(cls_probs_batch).mean(dim=0)
            avg_seg_preds = torch.stack(seg_preds_batch).mean(dim=0)

            all_probs.extend(avg_cls_probs.tolist())
            all_truths.extend(yb.cpu().tolist())
            all_seg_preds.extend(avg_seg_preds.numpy())
            all_seg_truths.extend(mb.cpu().numpy())

    return np.array(all_probs), np.array(all_truths), np.array(all_seg_preds), np.array(all_seg_truths)

def calculate_metrics(y_true, y_pred, y_probs):
    """Calculate raw and “padded” classification metrics."""
    # 1) Raw
    raw_accuracy  = accuracy_score(y_true, y_pred)
    raw_precision = precision_score(y_true, y_pred, average='weighted', zero_division=0)
    raw_recall    = recall_score(y_true, y_pred, average='weighted', zero_division=0)
    raw_f1        = f1_score(y_true, y_pred, average='weighted', zero_division=0)
    raw_mcc       = matthews_corrcoef(y_true, y_pred)

    # 2) Build metrics dict
    metrics = {
        'accuracy':  raw_accuracy,
        'precision': raw_precision,
        'recall':    raw_recall,
        'f1':        raw_f1,
        'mcc':       raw_mcc,
    }

    # 3) ROC-AUC and PR-AUC (only if both classes present)
    if len(np.unique(y_true)) > 1:
        roc_auc = roc_auc_score(y_true, y_probs)
        prec_vals, rec_vals, _ = precision_recall_curve(y_true, y_probs)
        pr_auc = auc(rec_vals, prec_vals)
        metrics.update({
            'roc_auc': roc_auc,
            'pr_auc':  pr_auc
        })

    return metrics

def calculate_segmentation_metrics(y_true_seg, y_pred_seg, threshold=0.5):
    """Calculate segmentation metrics"""
    y_pred_binary = (y_pred_seg > threshold).astype(int)
    y_true_binary = (y_true_seg > threshold).astype(int)

    # Dice coefficient
    intersection = np.sum(y_pred_binary * y_true_binary)
    dice = (2.0 * intersection) / (np.sum(y_pred_binary) + np.sum(y_true_binary) + 1e-7)

    # IoU (Jaccard index)
    union = np.sum(y_pred_binary) + np.sum(y_true_binary) - intersection
    iou = intersection / (union + 1e-7)

    # Hausdorff distance (simplified version)
    try:
        if np.sum(y_pred_binary) > 0 and np.sum(y_true_binary) > 0:
            pred_points = np.column_stack(np.where(y_pred_binary > 0))
            true_points = np.column_stack(np.where(y_true_binary > 0))
            hausdorff = max(
                directed_hausdorff(pred_points, true_points)[0],
                directed_hausdorff(true_points, pred_points)[0]
            )
        else:
            hausdorff = float('inf')
    except:
        hausdorff = float('inf')

    return {
        'dice': dice,
        'iou': iou,
        'hausdorff': hausdorff if hausdorff != float('inf') else 100.0
    }

# 8. Validation Set Evaluation
print("\n=== Validation Set Evaluation ===")
val_probs, val_truths, val_seg_preds, val_seg_truths = ensemble_predict(cv_models, val_loader)
val_preds = (val_probs >= 0.5).astype(int)

# Classification metrics
val_metrics  = calculate_metrics(val_truths, val_preds, val_probs)
print("Validation Classification Metrics:")
for metric, value in val_metrics.items():
    print(f"  {metric.upper()}: {value:.4f}")

# Segmentation metrics
val_seg_metrics = calculate_segmentation_metrics(val_seg_truths.flatten(), val_seg_preds.flatten())
print("\nValidation Segmentation Metrics:")
for metric, value in val_seg_metrics.items():
    print(f"  {metric.upper()}: {value:.4f}")

# 9. Test Set Evaluation
print("\n=== Test Set Evaluation ===")
test_probs, test_truths, test_seg_preds, test_seg_truths = ensemble_predict(cv_models, test_loader)
test_preds = (test_probs >= 0.5).astype(int)

# Classification metrics
test_metrics = calculate_metrics(test_truths, test_preds, test_probs)
print("Test Classification Metrics:")
for metric, value in test_metrics.items():
    print(f"  {metric.upper()}: {value:.4f}")

# Segmentation metrics
test_seg_metrics = calculate_segmentation_metrics(test_seg_truths.flatten(), test_seg_preds.flatten())
print("\nTest Segmentation Metrics:")
for metric, value in test_seg_metrics.items():
    print(f"  {metric.upper()}: {value:.4f}")

# 10. Visualization Functions
def plot_roc_curve(y_true, y_probs, title="ROC Curve"):
    """Plot ROC curve"""
    fpr, tpr, _ = roc_curve(y_true, y_probs)
    auc_score = roc_auc_score(y_true, y_probs)

    plt.figure(figsize=(8, 6))
    plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {auc_score:.3f})')
    plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(title)
    plt.legend(loc="lower right")
    plt.grid(True, alpha=0.3)
    plt.show()

def plot_precision_recall_curve(y_true, y_probs, title="Precision-Recall Curve"):
    """Plot Precision-Recall curve"""
    precision_vals, recall_vals, _ = precision_recall_curve(y_true, y_probs)
    pr_auc = auc(recall_vals, precision_vals)

    plt.figure(figsize=(8, 6))
    plt.plot(recall_vals, precision_vals, color='blue', lw=2, label=f'PR curve (AUC = {pr_auc:.3f})')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('Recall')
    plt.ylabel('Precision')
    plt.title(title)
    plt.legend(loc="lower left")
    plt.grid(True, alpha=0.3)
    plt.show()

def plot_confusion_matrix(y_true, y_pred, classes, title="Confusion Matrix"):
    """Plot confusion matrix"""
    cm = confusion_matrix(y_true, y_pred)

    plt.figure(figsize=(8, 6))
    plt.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
    plt.title(title)
    plt.colorbar()

    # Add text annotations
    thresh = cm.max() / 2.
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(j, i, f'{cm[i, j]}',
                    horizontalalignment="center",
                    color="white" if cm[i, j] > thresh else "black",
                    fontsize=16, fontweight='bold')

    plt.xticks(range(len(classes)), classes)
    plt.yticks(range(len(classes)), classes)
    plt.xlabel('Predicted Label')
    plt.ylabel('True Label')
    plt.tight_layout()
    plt.show()

def visualize_predictions(models, dataset, num_samples=6):
    """Visualize model predictions on sample images"""
    model = models[0]  # Use first model for visualization
    model.eval()

    # Get random samples
    indices = random.sample(range(len(dataset)), min(num_samples, len(dataset)))

    fig, axes = plt.subplots(3, len(indices), figsize=(20, 12))
    if len(indices) == 1:
        axes = axes.reshape(-1, 1)

    with torch.no_grad():
        for idx, sample_idx in enumerate(indices):
            x, y_true, mask_true = dataset[sample_idx]
            x_batch = x.unsqueeze(0).to(DEVICE)

            # Get predictions
            ctx = autocast("cuda") if USE_AMP else autocast("cpu")
            with ctx:
                seg_logits, cls_logits = model(x_batch)

            # Process outputs
            cls_probs = torch.softmax(cls_logits, dim=1)[0]
            y_pred = cls_probs.argmax().item()
            confidence = cls_probs.max().item()

            seg_pred = torch.sigmoid(seg_logits).squeeze().cpu().numpy()

            # Convert input back to displayable format
            img = x[:3].permute(1, 2, 0).cpu().numpy()
            img = (img * np.array(STD[:3]) + np.array(MEAN[:3])).clip(0, 1)

            # Plot original image
            axes[0, idx].imshow(img)
            axes[0, idx].set_title(f'Original\nTrue: {CLASSES[y_true]}\nPred: {CLASSES[y_pred]} ({confidence:.3f})')
            axes[0, idx].axis('off')

            # Plot true mask
            axes[1, idx].imshow(mask_true.cpu().numpy(), cmap='gray')
            axes[1, idx].set_title('True Mask')
            axes[1, idx].axis('off')

            # Plot predicted mask
            axes[2, idx].imshow(seg_pred, cmap='gray')
            axes[2, idx].set_title('Predicted Mask')
            axes[2, idx].axis('off')

    plt.tight_layout()
    plt.show()

# 11. Generate Visualizations
print("\n=== Generating Visualizations ===")

# ROC Curves
if len(np.unique(test_truths)) > 1:
    plot_roc_curve(test_truths, test_probs, "Test Set ROC Curve")
    plot_precision_recall_curve(test_truths, test_probs, "Test Set Precision-Recall Curve")

# Confusion Matrices
plot_confusion_matrix(test_truths, test_preds, CLASSES, "Test Set Confusion Matrix")

# Sample predictions
test_dataset = ThyroidDataset(test_p, test_l, augment=False)
visualize_predictions(cv_models, test_dataset, num_samples=6)

# 12. Save Results
print("\n=== Saving Results ===")

# Compile all results
final_results = {
    'validation_metrics': {
        'classification': val_metrics,
        'segmentation': val_seg_metrics
    },
    'test_metrics': {
        'classification': test_metrics,
        'segmentation': test_seg_metrics
    },
    'dataset_info': {
        'total_images': len(paths),
        'train_images': len(train_p),
        'val_images': len(val_p),
        'test_images': len(test_p),
        'class_distribution': {
            'benign': labels.count(0),
            'malignant': labels.count(1)
        }
    },
    'model_info': {
        'architecture': 'HybridSwinCSASN',
        'num_folds': len(cv_models),
        'ensemble_method': 'average'
    }
}

# Save results to JSON
results_file = os.path.join(OUT_ROOT, 'final_results.json')
with open(results_file, 'w') as f:
    json.dump(final_results, f, indent=2)

print(f"Results saved to: {results_file}")

# Save predictions
predictions_df = pd.DataFrame({
    'image_path': test_p,
    'true_label': test_truths,
    'predicted_label': test_preds,
    'predicted_probability': test_probs,
    'correct_prediction': test_truths == test_preds
})

predictions_file = os.path.join(OUT_ROOT, 'test_predictions.csv')
predictions_df.to_csv(predictions_file, index=False)
print(f"Test predictions saved to: {predictions_file}")

# 13. Performance Summary
print("\n" + "="*60)
print("FINAL PERFORMANCE SUMMARY")
print("="*60)

print(f"\nDataset Statistics:")
print(f"  Total Images: {len(paths)}")
print(f"  Train/Val/Test Split: {len(train_p)}/{len(val_p)}/{len(test_p)}")
print(f"  Class Distribution: Benign {labels.count(0)}, Malignant {labels.count(1)}")

print(f"\nTest Set Classification Performance:")
print(f"  Accuracy:  {test_metrics['accuracy']:.4f}")
print(f"  Precision: {test_metrics['precision']:.4f}")
print(f"  Recall:    {test_metrics['recall']:.4f}")
print(f"  F1-Score:  {test_metrics['f1']:.4f}")
print(f"  ROC-AUC:   {test_metrics.get('roc_auc', 'N/A')}")
print(f"  MCC:       {test_metrics['mcc']:.4f}")

print(f"\nTest Set Segmentation Performance:")
print(f"  Dice Coefficient: {test_seg_metrics['dice']:.4f}")
print(f"  IoU:              {test_seg_metrics['iou']:.4f}")
print(f"  Hausdorff Dist:   {test_seg_metrics['hausdorff']:.2f}")

print("\n" + "="*60)
print("Analysis completed successfully!")
print(f"All results and models saved in: {OUT_ROOT}")
print("="*60)

In [ ]:
import os
import random
import cv2
import numpy as np
import torch
import matplotlib.pyplot as plt

# --- 1. Constants & paths (same as above) ---
MEAN = np.array([0.485, 0.456, 0.406, 0.5], dtype=np.float32)
STD  = np.array([0.229, 0.224, 0.225, 0.2], dtype=np.float32)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
CLASSES = ['Benign', 'Malignant']
IMG_OUT = os.path.join(DS_ROOT, 'preprocessed_images')
OUT_ROOT = '/content/drive/MyDrive/Thyroid_Results_SWIN_CSASN'

# assume cv_models is your list of trained models
model = cv_models[0].to(DEVICE)
model.eval()

# --- 2. Pick one random test image ---
# test_p is your list of test file paths
img_path = random.choice(test_p)
print("Selected:", img_path)

# --- 3. Preprocess exactly as in train/val ---
# 3.1 load & resize
orig = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB) / 255.0
resized = cv2.resize(orig, (224, 224))

# 3.2 Otsu mask
gray = cv2.cvtColor((resized*255).astype(np.uint8), cv2.COLOR_RGB2GRAY)
_, mask = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
mask = mask.astype(np.float32) / 255.0

# 3.3 stack & normalize
x = np.dstack([resized, mask])
x = (x - MEAN) / STD
x_tensor = torch.tensor(x, dtype=torch.float32).permute(2,0,1).unsqueeze(0).to(DEVICE)

# --- 4. Inference ---
with torch.no_grad():
    seg_logits, cls_logits = model(x_tensor)
    # classification
    probs = torch.softmax(cls_logits, dim=1)[0].cpu().numpy()
    cls_idx = probs.argmax()
    cls_prob = probs[cls_idx]
    # segmentation
    seg_prob = torch.sigmoid(seg_logits)[0,0].cpu().numpy()

print(f"Classified as {CLASSES[cls_idx]} with probability {cls_prob:.3f}")

# --- 5. Visualization (styled to match your example) ---
import matplotlib.pyplot as plt

# Prepare save path
pred_dir = os.path.join(OUT_ROOT, 'predictions')
os.makedirs(pred_dir, exist_ok=True)
save_path = os.path.join(pred_dir, f"randpred_{os.path.basename(img_path)}")

# Create figure with 3 side-by-side panels
fig, axes = plt.subplots(1, 3, figsize=(18, 6), tight_layout=True)

# Panel 1: Original
axes[0].imshow(resized)
axes[0].set_title(f"Original – Pred: {CLASSES[cls_idx]} ({cls_prob:.2f})", fontsize=16)
axes[0].axis('off')

# Panel 2: Input Otsu Mask
axes[1].imshow(mask, cmap='gray')
axes[1].set_title("Input Otsu Mask", fontsize=16)
axes[1].axis('off')

# Panel 3: Predicted Segmentation
axes[2].imshow(seg_prob, cmap='inferno')
axes[2].set_title("Predicted Segmentation", fontsize=16)
axes[2].axis('off')

# Save at high-res and show
fig.savefig(save_path, dpi=300, bbox_inches='tight', pad_inches=0.1)
plt.show()

print(f"Prediction figure saved to:\n  {save_path}")